<a href="https://colab.research.google.com/github/PrudhviNallagatla/Advanced-Recognition-of-License-Plates-ARLP/blob/main/src/ARLP-YOLOv8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!rm -rf /content/dataset/dataset

In [4]:
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/Elearnmarkets/dataset.zip -d /content

In [6]:
%pip install ultralytics

import os
import shutil
import xml.etree.ElementTree as ET
from glob import glob
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Data Conversion (XML to YOLO Format)

In [9]:
def convert_xml_to_yolo(xml_file, output_txt_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    size = root.find('size')
    w_img = float(size.find('width').text)
    h_img = float(size.find('height').text)

    with open(output_txt_file, 'w') as f:
        for obj in root.iter('object'):

            xmlbox = obj.find('bndbox')
            xmin = float(xmlbox.find('xmin').text)
            ymin = float(xmlbox.find('ymin').text)
            xmax = float(xmlbox.find('xmax').text)
            ymax = float(xmlbox.find('ymax').text)

            # Calculate YOLO coordinates (normalized)
            x_center = ((xmin + xmax) / 2) / w_img
            y_center = ((ymin + ymax) / 2) / h_img
            width = (xmax - xmin) / w_img
            height = (ymax - ymin) / h_img

            # Class is 0 (license plate)
            f.write(f"0 {x_center} {y_center} {width} {height}\n")

def process_dataset(data_dir):
    os.makedirs(f"yolo_dataset/images/{data_dir}", exist_ok=True)
    os.makedirs(f"yolo_dataset/labels/{data_dir}", exist_ok=True)

    original_dir = f"/content/dataset/ARLP Datasets ver-2/{data_dir}"
    xml_files = glob(os.path.join(original_dir, '*.xml'))

    for xml_file in xml_files:
        base_name = os.path.basename(xml_file).replace('.xml', '')

        # Find corresponding image
        img_file = os.path.join(original_dir, f"{base_name}.jpg")
        if not os.path.exists(img_file):
            img_file = os.path.join(original_dir, f"{base_name}.png")

        if os.path.exists(img_file):
            # Copy image to YOLO directory
            shutil.copy(img_file, f"yolo_dataset/images/{data_dir}/{os.path.basename(img_file)}")

            # Convert and save label
            output_txt = f"yolo_dataset/labels/{data_dir}/{base_name}.txt"
            convert_xml_to_yolo(xml_file, output_txt)


In [10]:
# Run the conversion once
print("Processing training data...")
process_dataset('train')
print("Processing test data...")
process_dataset('test')
print("Data converted successfully!")

Processing training data...
Processing test data...
Data converted successfully!


### create YOLO config (data.yaml)

In [13]:
yaml_content = """
path: /content/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/test  # val images (relative to 'path')

# Classes
names:
  0: license_plate
"""

with open('yolo_dataset/data.yaml', 'w') as f:
    f.write(yaml_content)
print("data.yaml created.")

data.yaml created.


### training

In [14]:
# Load YOLOv8 nano model
model = YOLO('yolov8n.pt')

model.train(data='yolo_dataset/data.yaml', epochs=100, imgsz=640, patience=15, batch=16)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, per

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f2e5e455dc0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [15]:
!cp -r /content/runs /content/drive/MyDrive/Elearnmarkets/
!cp -r /content/yolo26n.pt /content/drive/MyDrive/Elearnmarkets/